# 01_06: Translation & Multilingual Models

**Objective:** Learn to translate text between languages using seq2seq models and explore multilingual NLP capabilities.

**Prerequisites:** Basic Python, familiarity with HuggingFace pipelines (Notebook 00_03). Understanding of encoder-decoder architecture (Notebook 00_02) helpful.

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | Helsinki-NLP/opus-mt-en-fr | ~300MB | 4GB RAM | Any CPU | English→French |
| **Medium (CPU)** | Helsinki-NLP/opus-mt-en-de | ~300MB | 4GB RAM | Any CPU | English→German |
| **Large (GPU)** | facebook/nllb-200-distilled-600M | ~600MB | 8GB RAM | RTX 4080+ | 200 languages |

## Expected Behaviors

### First Time Running
- **Model download**: ~300MB per MarianMT model (1-2 minutes each)
- NLLB-200 is ~600MB if used
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

### What You'll See
- MarianMT models handle one language pair each (e.g., en→fr)
- Translations are fluent for common language pairs
- NLLB-200 can translate between 200 languages in any direction

### Common Observations
- Short sentences translate well; complex sentences may lose nuance
- Technical/domain-specific terms may not translate accurately
- Languages with more training data (French, German, Spanish) get better results

## Overview

**Machine translation** is a sequence-to-sequence (seq2seq) task: the model reads text in one language and generates equivalent text in another.

### Translation Model Families

| Model Family | Developer | Languages | Architecture | Size |
|-------------|-----------|-----------|--------------|------|
| **MarianMT** | Helsinki-NLP | 1000+ language pairs | Encoder-Decoder | ~300MB each |
| **NLLB-200** | Meta | 200 languages | Encoder-Decoder | 600MB-3.3GB |
| **mBART** | Meta | 50 languages | Encoder-Decoder | ~600MB |
| **M2M-100** | Meta | 100 languages | Encoder-Decoder | 480MB-12GB |

We'll primarily use **MarianMT** models from the Helsinki-NLP project. Each model handles one language pair and is small enough to run on CPU. For multilingual scenarios, we'll also explore **NLLB-200**.

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    MarianMTModel,
    MarianTokenizer,
)

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

MarianMT models follow the naming pattern `Helsinki-NLP/opus-mt-{src}-{tgt}`. Each model is specialized for one language direction.

In [ ]:
# English → French (default)
EN_FR_MODEL = 'Helsinki-NLP/opus-mt-en-fr'

# Additional language pairs
EN_DE_MODEL = 'Helsinki-NLP/opus-mt-en-de'   # English → German
EN_ES_MODEL = 'Helsinki-NLP/opus-mt-en-es'   # English → Spanish
FR_EN_MODEL = 'Helsinki-NLP/opus-mt-fr-en'   # French → English

# Multilingual (200 languages, GPU recommended)
# NLLB_MODEL = 'facebook/nllb-200-distilled-600M'

## Method 1: Pipeline API

The translation pipeline is the fastest way to translate text. You specify the model (which determines the language pair), and it handles tokenization, generation, and decoding automatically.

In [ ]:
# Create English → French translator
translator_en_fr = pipeline(
    'translation',
    model=EN_FR_MODEL,
    device=device,
)

# Translate sample sentences
english_texts = [
    'Hello, how are you?',
    'The weather is beautiful today.',
    'Machine learning is transforming the world.',
    'I would like to book a table for two people.',
    'Artificial intelligence will change how we work and live.',
]

translations = translator_en_fr(english_texts)

translation_df = pd.DataFrame([{
    'English': en,
    'French': tr['translation_text'],
} for en, tr in zip(english_texts, translations)])
translation_df

The translations are generally fluent and accurate. MarianMT models are trained on the OPUS parallel corpus, which is one of the largest open-source collections of translated text. Let's try more language pairs to see the breadth of coverage.

In [ ]:
# Translate to multiple languages
def translate_to_multiple(
    text: str,
    model_names: dict[str, str],
) -> pd.DataFrame:
    """Translate a text into multiple languages.

    Args:
        text: English text to translate.
        model_names: Dict mapping language names to model identifiers.

    Returns:
        DataFrame with translations in each language.
    """
    results: list[dict[str, str]] = [{'Language': 'English (original)', 'Translation': text}]
    
    for language, model_name in model_names.items():
        try:
            translator = pipeline('translation', model=model_name, device=device)
            result = translator(text)
            results.append({
                'Language': language,
                'Translation': result[0]['translation_text'],
            })
        except Exception as error:
            results.append({
                'Language': language,
                'Translation': f'Error: {error}',
            })
    
    return pd.DataFrame(results)


print('=== Multi-Language Translation ===')
translate_to_multiple(
    'The future of artificial intelligence is both exciting and challenging.',
    {
        'French': EN_FR_MODEL,
        'German': EN_DE_MODEL,
        'Spanish': EN_ES_MODEL,
    }
)

## Method 2: Manual Model Loading

For more control over the translation process, we can load the model manually. This allows us to adjust generation parameters like beam search width, length penalties, and sampling strategies.

In [ ]:
# Load MarianMT model manually
mt_tokenizer = MarianTokenizer.from_pretrained(EN_FR_MODEL)
mt_model = MarianMTModel.from_pretrained(EN_FR_MODEL).to(device)
mt_model.eval()

print(f'Model: {EN_FR_MODEL}')
print(f'Parameters: {sum(p.numel() for p in mt_model.parameters()):,}')
print(f'Vocab size: {mt_tokenizer.vocab_size}')
print(f'Supported languages: {mt_tokenizer.supported_language_codes}')

In [ ]:
def translate_with_options(
    text: str,
    tokenizer: transformers.PreTrainedTokenizer,
    model: torch.nn.Module,
    num_beams: int = 4,
    max_length: int = 128,
    num_return_sequences: int = 1,
) -> list[str]:
    """Translate text with configurable generation parameters.

    Args:
        text: Text to translate.
        tokenizer: The translation model's tokenizer.
        model: The translation model.
        num_beams: Number of beams for beam search.
        max_length: Maximum output length in tokens.
        num_return_sequences: Number of translations to return.

    Returns:
        List of translated strings.
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            early_stopping=True,
        )
    
    translations = [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]
    return translations


# Compare different beam widths
test_text = 'Deep learning models require large amounts of data and compute.'
print(f'Original: "{test_text}"\n')

beam_results: list[dict[str, str]] = []
for num_beams in [1, 4, 8]:
    translation = translate_with_options(
        test_text, mt_tokenizer, mt_model, num_beams=num_beams
    )
    beam_results.append({
        'Beam Width': str(num_beams),
        'Translation': translation[0],
        'Method': 'Greedy' if num_beams == 1 else f'Beam Search ({num_beams})',
    })

pd.DataFrame(beam_results)

Beam search explores multiple translation hypotheses simultaneously and typically produces more fluent translations than greedy decoding (beam width = 1). However, increasing beam width beyond 4-8 usually offers diminishing returns while increasing computation time.

## Round-Trip Translation Quality Check

A simple way to assess translation quality is **round-trip translation**: translate to the target language and back. If the back-translation closely matches the original, the translation is likely accurate. This isn't perfect, but it's a useful quick check.

In [ ]:
def round_trip_translation(
    text: str,
    forward_model: str,
    backward_model: str,
) -> dict[str, str]:
    """Perform round-trip translation and compare with original.

    Args:
        text: Original text.
        forward_model: Model for source→target translation.
        backward_model: Model for target→source translation.

    Returns:
        Dictionary with original, translated, and back-translated text.
    """
    forward_pipe = pipeline('translation', model=forward_model, device=device)
    backward_pipe = pipeline('translation', model=backward_model, device=device)
    
    translated = forward_pipe(text)[0]['translation_text']
    back_translated = backward_pipe(translated)[0]['translation_text']
    
    return {
        'Original': text,
        'Translated (FR)': translated,
        'Back-translated (EN)': back_translated,
    }


round_trip_texts = [
    'The quick brown fox jumps over the lazy dog.',
    'Please send me the report by Friday.',
    'Quantum computing will revolutionize cryptography.',
    'She sells seashells by the seashore.',
]

print('=== Round-Trip Translation (EN → FR → EN) ===')
round_trip_results = [round_trip_translation(t, EN_FR_MODEL, FR_EN_MODEL) for t in round_trip_texts]
pd.DataFrame(round_trip_results)

Round-trip translation reveals interesting patterns: simple, common sentences survive the round trip well, while idioms, puns, and culturally-specific expressions often get distorted. This is because translation models learn statistical patterns from parallel text, and rare constructions have fewer training examples.

## Practical Applications

In [ ]:
# Application: Translate a structured document
def translate_document(
    paragraphs: list[str],
    translator_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Translate a multi-paragraph document.

    Args:
        paragraphs: List of paragraph strings.
        translator_pipe: HuggingFace translation pipeline.

    Returns:
        DataFrame with source and translated paragraphs.
    """
    results: list[dict[str, str]] = []
    for i, para in enumerate(paragraphs):
        translation = translator_pipe(para)[0]['translation_text']
        results.append({
            'Paragraph': str(i + 1),
            'English': para[:80] + ('...' if len(para) > 80 else ''),
            'French': translation[:80] + ('...' if len(translation) > 80 else ''),
        })
    return pd.DataFrame(results)


document = [
    'Welcome to our company. We are a leading provider of AI solutions.',
    'Our team consists of experts in machine learning, natural language processing, and computer vision.',
    'We offer consulting services, custom model development, and production deployment support.',
    'Contact us to learn how AI can transform your business.',
]

print('=== Document Translation ===')
translate_document(document, translator_en_fr)

## Performance Benchmarking

Translation involves both encoding and decoding (autoregressive generation), so it's typically slower than classification tasks. Let's measure the relationship between text length and latency.

In [ ]:
def benchmark_translation(
    texts: list[str],
    translator_pipe: transformers.Pipeline,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark translation speed across different text lengths.

    Args:
        texts: List of texts of varying lengths.
        translator_pipe: HuggingFace translation pipeline.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results.
    """
    results: list[dict[str, str]] = []
    
    for text in texts:
        word_count = len(text.split())
        
        # Warmup
        translator_pipe(text)
        
        times: list[float] = []
        for _ in range(num_runs):
            start = time.perf_counter()
            translator_pipe(text)
            times.append(time.perf_counter() - start)
        
        results.append({
            'Input Words': str(word_count),
            'Mean Latency (ms)': f'{np.mean(times) * 1000:.1f}',
            'Std (ms)': f'{np.std(times) * 1000:.1f}',
            'Words/sec': f'{word_count / np.mean(times):.0f}',
            'Device': str(device),
        })
    
    return pd.DataFrame(results)


bench_texts = [
    'Hello world.',
    'Machine learning models are becoming increasingly powerful and accessible.',
    'The development of large language models has transformed natural language processing. '
    'These models can now perform translation, summarization, question answering, and '
    'many other tasks with impressive accuracy. However, they require significant '
    'computational resources for both training and inference.',
]

print('=== Translation Speed vs Text Length ===')
benchmark_translation(bench_texts, translator_en_fr)

## Exercises

1. **Language chain**: Translate a sentence through 3+ languages (EN→FR→DE→EN) and see how much meaning is preserved. Try with both simple and complex sentences.

2. **Domain-specific translation**: Test the translator on technical text (medical, legal, programming). Where does it struggle?

3. **NLLB exploration**: If you have GPU access, load `facebook/nllb-200-distilled-600M` and translate between less common language pairs (e.g., Swahili↔Japanese).

4. **Batch optimization**: Compare translating 10 sentences one-by-one vs batched. Is there a speed difference with the pipeline?

## Key Takeaways

- **MarianMT** provides lightweight (~300MB) models for 1000+ language pairs via Helsinki-NLP
- Translation uses **encoder-decoder** architecture: the encoder reads the source, the decoder generates the target autoregressively
- **Beam search** (width 4-8) generally produces better translations than greedy decoding
- **Round-trip translation** is a quick quality check — translate to target and back, then compare with the original
- For multilingual scenarios (many language pairs), consider **NLLB-200** instead of loading separate models per pair

## Next Steps & Resources

**Next notebook**: [01_07 — Fine-tuning with Unsloth](01_07_nlp_fine_tuning_unsloth.ipynb) — learn to fine-tune language models efficiently.

**Resources:**
- [Helsinki-NLP MarianMT Models](https://huggingface.co/Helsinki-NLP) — 1000+ language pair models
- [NLLB-200](https://huggingface.co/facebook/nllb-200-distilled-600M) — Meta's 200-language model
- [OPUS Parallel Corpus](https://opus.nlpl.eu/) — Training data source for MarianMT
- [Translation Models on Hub](https://huggingface.co/models?pipeline_tag=translation)